In [1]:
import importlib
import sys

# Remove cached modules
if 'github' in sys.modules:
    del sys.modules['github']
if 'github.utils' in sys.modules:
    del sys.modules['github.utils']
if 'github.utils.validation' in sys.modules:
    del sys.modules['github.utils.validation']

# Now re-import
from utils.validation import train_distance_models, load_all_keypoints

c:\Users\rwcki\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'train_distance_models' from 'utils.validation' (d:\Politechnika\Deadlift_publikacja_nr_2\Skrypty\github\utils\validation\__init__.py)

In [1]:
import sys
sys.path.insert(0,r"D:\Politechnika\Deadlift_publikacja_nr_2\Skrypty\github")
%pip install -e D:\Politechnika\Deadlift_publikacja_nr_2\Skrypty\github

import os
os.chdir(r"D:\Politechnika\Deadlift_publikacja_nr_2\Skrypty\github")

Obtaining file:///D:/Politechnika/Deadlift_publikacja_nr_2/Skrypty/github
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for utils-yolo-validation (pyproject.toml): started
  Building editable for utils-yolo-validation (pyproject.toml): finished with status 'done'
  Created wheel for utils-yolo-validation: filename=utils_yolo_validation-0.1.0-0.editable-py3-none-any.whl size=9885 sha256=0f76d67950930c88b83864fbd90da6d23e16454d5920fce42ae19b17b6558413
  Stored in directory: C:\Users\rwcki\AppData\Local\Temp\pip-ephem-wheel-cac

In [ ]:
from utils.validation import train_distance_models, load_all_keypoints, distance_matrix_from_keypoints, predict_distances

In [ ]:
keypoint_list = load_all_keypoints(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\train.txt")
distance_matrix = distance_matrix_from_keypoints(keypoint_list)
models = train_distance_models(distance_matrix)

In [14]:
# Check dataset info
print(f"Number of poses: {len(keypoint_list)}")
print(f"Distance matrix shape: {distance_matrix.shape}")
print(f"Number of models trained: {len(models)}")

# If we have interior distances, predict them
if len(models) > 0:
    predictions = predict_distances(models, distance_matrix)
    print(f"\nPredictions shape: {predictions.shape}")
    print(f"First prediction: {predictions[0]}")
else:
    print("\nNo models trained (need at least 3 distances per pose)")
    # Try with exclude_endpoints=False to predict all distances
    models_all = train_distance_models(distance_matrix, exclude_endpoints=False)
    print(f"Models trained (all distances): {len(models_all)}")
    predictions = predict_distances(models_all, distance_matrix, exclude_endpoints=False)
    print(f"Predictions shape: {predictions.shape}")
    print(f"First 5 predictions:\n{predictions[0]}")

Number of poses: 4758
Distance matrix shape: (4758, 8)
Number of models trained: 8

Predictions shape: (4758, 8)
First prediction: [0.03253533 0.04016448 0.05010252 0.04655484 0.05120122 0.04989185
 0.04724216 0.04595231]


In [15]:
# Check keypoint_list contents
print(f"Type: {type(keypoint_list)}")
print(f"Length (number of poses): {len(keypoint_list)}")
print(f"First pose shape: {keypoint_list[0].shape}")
print(f"First pose:\n{keypoint_list[0]}")

# Check a few more examples
print(f"\nSecond pose shape: {keypoint_list[1].shape}")
print(f"All pose shapes: {[kp.shape for kp in keypoint_list[:5]]}")

Type: <class 'list'>
Length (number of poses): 4758
First pose shape: (9, 2)
First pose:
[[0.338702 0.354567]
 [0.338077 0.37875 ]
 [0.388437 0.386106]
 [0.430769 0.395721]
 [0.481082 0.408486]
 [0.529639 0.421274]
 [0.57988  0.434375]
 [0.630481 0.447572]
 [0.661322 0.467957]]

Second pose shape: (9, 2)
All pose shapes: [(9, 2), (9, 2), (9, 2), (9, 2), (9, 2)]


In [8]:
distance_matrix

array([[0.02419108, 0.0508944 , 0.04341021, ..., 0.05192104, 0.05229361,
        0.03696912],
       [0.01432873, 0.0194283 , 0.01994421, ..., 0.01737636, 0.02007943,
        0.01721844],
       [0.0164054 , 0.01722845, 0.01823102, ..., 0.02404444, 0.02031984,
        0.01812095],
       ...,
       [0.04519376, 0.08406599, 0.07423233, ..., 0.05882503, 0.05667059,
        0.04510959],
       [0.0246832 , 0.02377226, 0.02567678, ..., 0.02933621, 0.02822039,
        0.02449138],
       [0.02991768, 0.03824821, 0.04184045, ..., 0.04654787, 0.04221094,
        0.04076834]])

In [16]:
from utils.validation import create_prediction_report

# Get predictions
predictions = predict_distances(models, distance_matrix)

# Create sorted report (descending by error)
report = create_prediction_report(
    predictions=predictions,
    labels=distance_matrix,
    sample_ids=[f"image_{i}.jpg" for i in range(len(distance_matrix))],
    metric="mae"  # or "mse", "rmse", "max"
)

# Print top 10 worst predictions
for item in report[:10]:
    print(f"Rank {item['rank']}: {item['sample_id']} - error: {item['error']:.6f}")
    print(f"  Predictions: {item['predictions']}")
    print(f"  Labels:      {item['labels']}")
    print(f"  Differences: {item['differences']}\n")

Rank 1: image_901.jpg - error: 0.066272
  Predictions: [0.09108557713636198, 0.09076053710889934, 0.15324796933592982, 0.096120922499737, 0.15123676325481458, 0.12683041477270152, 0.09301821011468528, 0.06846344989170203]
  Labels:      [0.06449514105729204, 0.10146095724464661, 0.10765428984485474, 0.25160378293658464, 0.07664649931992978, 0.05592786568429014, 0.0819816663712077, 0.20374579182157354]
  Differences: [0.02659043607906994, -0.010700420135747277, 0.04559367949107508, -0.15548286043684764, 0.0745902639348848, 0.07090254908841137, 0.011036543743477581, -0.1352823419298715]

Rank 2: image_254.jpg - error: 0.057805
  Predictions: [0.05038076624586366, 0.08412065847645803, 0.0837893007282248, 0.11329119101037236, 0.14287892641507016, 0.08125494850576384, 0.16664600016741563, 0.11215581619238821]
  Labels:      [0.07586235662698601, 0.06652553369195917, 0.0904219584503676, 0.07837398860591438, 0.04965720978871044, 0.2670560888989427, 0.0958021602470425, 0.08420736472542045]
  D

In [3]:
from utils.datasets import YOLOPoseDataset
dataset = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\train.txt")

print(f"Dataset length: {len(dataset)}")
if len(dataset) > 0:
    print(f"First item: {dataset[0]}")

Dataset length: 3897
First item: YOLOSample(image=array([[[92, 97, 96],
        [92, 97, 96],
        [92, 97, 96],
        ...,
        [38, 43, 42],
        [38, 43, 42],
        [38, 43, 42]],

       [[85, 90, 89],
        [85, 90, 89],
        [85, 90, 89],
        ...,
        [38, 43, 42],
        [38, 43, 42],
        [38, 43, 42]],

       [[86, 88, 88],
        [85, 87, 87],
        [85, 87, 87],
        ...,
        [38, 43, 42],
        [38, 43, 42],
        [38, 43, 42]],

       ...,

       [[70, 73, 77],
        [76, 79, 83],
        [80, 83, 87],
        ...,
        [53, 57, 58],
        [53, 57, 58],
        [53, 57, 58]],

       [[71, 74, 78],
        [71, 74, 78],
        [72, 75, 79],
        ...,
        [56, 60, 61],
        [55, 59, 60],
        [55, 59, 60]],

       [[72, 75, 79],
        [67, 70, 74],
        [64, 67, 71],
        ...,
        [58, 62, 63],
        [57, 61, 62],
        [57, 61, 62]]], dtype=uint8), keypoints=array([[0.4997  , 0.411262, 0.3

In [12]:
# Debug: Check what's in the dataset directory
import os

train_txt = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\train.txt"
dataset_dir = os.path.dirname(train_txt)

print(f"Dataset directory: {dataset_dir}")
print(f"Dataset dir exists: {os.path.exists(dataset_dir)}")

# Read contents of train.txt
with open(train_txt, 'r') as f:
    lines = f.readlines()
    
print(f"\nFist 3 lines from train.txt:")
for i, line in enumerate(lines[:3]):
    print(f"  {i}: {line.strip()}")

# Check if images and labels directories exist
images_dir = os.path.join(dataset_dir, 'images')
labels_dir = os.path.join(dataset_dir, 'labels')

print(f"\nImages dir: {images_dir}")
print(f"  Exists: {os.path.exists(images_dir)}")
if os.path.exists(images_dir):
    print(f"  Contents: {os.listdir(images_dir)[:3]}")

print(f"\nLabels dir: {labels_dir}")
print(f"  Exists: {os.path.exists(labels_dir)}")
if os.path.exists(labels_dir):
    print(f"  Contents: {os.listdir(labels_dir)[:3]}")

Dataset directory: D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged
Dataset dir exists: True

Fist 3 lines from train.txt:
  0: images/-2022-03-12-233600_png_jpg.rf.73596586d3a74eaa78b44a93aacb4c3b.jpg
  1: images/-2022-03-12-233624_png_jpg.rf.54558d2aef9b8a7c7c6d160b1cb155db.jpg
  2: images/-2022-03-12-233644_png_jpg.rf.4f6960b18d9a1a3ede45495ab3eca816.jpg

Images dir: D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\images
  Exists: True
  Contents: ['-2022-03-12-233600_png_jpg.rf.73596586d3a74eaa78b44a93aacb4c3b.jpg', '-2022-03-12-233624_png_jpg.rf.54558d2aef9b8a7c7c6d160b1cb155db.jpg', '-2022-03-12-233644_png_jpg.rf.4f6960b18d9a1a3ede45495ab3eca816.jpg']

Labels dir: D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\labels
  Exists: True
  Contents: ['-2022-03-12-233600_png_jpg.rf.73596586d3a74eaa78b44a93aacb4c3b.txt', '-2022-03-12-233624_png_jpg.rf.54558d2aef9b8a7c7c6d160b1cb155db.txt', '-2022-03-12-23

In [1]:
# Debug: Test loading keypoints directly
import sys
sys.path.insert(0, r"D:\Politechnika\Deadlift_publikacja_nr_2\Skrypty\github")

from utils.validation.image_validation import YPImageValidation

label_path = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\labels\-2022-03-12-233600_png_jpg.rf.73596586d3a74eaa78b44a93aacb4c3b.txt"

print(f"Label path exists: {__import__('os').path.exists(label_path)}")

try:
    kpts = YPImageValidation.load_keypoints(label_path)
    print(f"✅ Keypoints loaded!")
    print(f"Shape: {kpts.shape}")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

c:\Users\rwcki\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Label path exists: True
✅ Keypoints loaded!
Shape: (10, 3)


In [4]:
# Demonstrate YOLOSample with synthetic data
import numpy as np
from utils.datasets.yolo_pose_dataset import YOLOSample

# Create a sample object
sample = YOLOSample(
    image=np.ones((416, 416, 3), dtype=np.uint8),
    keypoints=np.random.rand(9, 3),
    image_path="/path/to/image.jpg",
    label_path="/path/to/label.txt"
)

# ✅ Attribute access (new way - cleaner!)
print("✅ ATTRIBUTE ACCESS (new):")
print(f"  sample.image shape: {sample.image.shape}")
print(f"  sample.keypoints shape: {sample.keypoints.shape}")
print(f"  sample.image_path: {sample.image_path}")

# ✅ Dict-like access (backward compatible)
print("\n✅ DICT-LIKE ACCESS (old, still works):")
print(f"  sample['image'] shape: {sample['image'].shape}")
print(f"  sample['keypoints'] shape: {sample['keypoints'].shape}")
print(f"  sample['image_path']: {sample['image_path']}")

print("\n✅ Both syntaxes access the SAME data!")

✅ ATTRIBUTE ACCESS (new):
  sample.image shape: (416, 416, 3)
  sample.keypoints shape: (9, 3)
  sample.image_path: /path/to/image.jpg

✅ DICT-LIKE ACCESS (old, still works):
  sample['image'] shape: (416, 416, 3)
  sample['keypoints'] shape: (9, 3)
  sample['image_path']: /path/to/image.jpg

✅ Both syntaxes access the SAME data!


In [5]:
# Test with REAL dataset
sample = dataset[0]

print("✅ ATTRIBUTE ACCESS (from real dataset):")
print(f"  sample.image shape: {sample.image.shape}")
print(f"  sample.keypoints shape: {sample.keypoints.shape}")
print(f"  sample.image_path: {sample.image_path[:50]}...")

print("\n✅ DICT-LIKE ACCESS (backward compatible):")
print(f"  sample['image'] shape: {sample['image'].shape}")
print(f"  sample['keypoints'] shape: {sample['keypoints'].shape}")
print(f"  sample['image_path']: {sample['image_path'][:50]}...")

✅ ATTRIBUTE ACCESS (from real dataset):
  sample.image shape: (416, 416, 3)
  sample.keypoints shape: (10, 3)
  sample.image_path: D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_t...

✅ DICT-LIKE ACCESS (backward compatible):
  sample['image'] shape: (416, 416, 3)
  sample['keypoints'] shape: (10, 3)
  sample['image_path']: D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_t...
